# LES turbulent flow past a sphere

Run this notebook to generate the simulation outputs:
- mid-plane velocity & pressure
- wake centerline profile
- Smagorinsky eddy viscosity map

## Boundary conditions
- **Inlet**: fixed $U_\infty$ + mild turbulence intensity
- **Outlet**: convective (Orlanski) + soft pressure (non-reflecting)
- **Outer walls**: free-slip
- **Sphere**: no-slip

In [ ]:
import os
print("cwd:", os.getcwd())

In [ ]:
import numpy as np
import meshio

mesh_npz = np.load("processed_mesh.npz")
if "boundary_tag" not in mesh_npz.files:
    print("Adding boundary tags to processed_mesh.npz ...")
    unique_faces = mesh_npz["unique_faces"]
    face_lookup = {tuple(f): i for i, f in enumerate(unique_faces)}
    m = meshio.read("fluid_mesh_3d.msh")
    names = {1: "inlet", 2: "outlet", 3: "walls", 4: "object"}
    faces = {k: [] for k in names.values()}
    boundary_tag = np.full(len(unique_faces), 5, dtype=np.int32)
    for block, tags in zip(m.cells[:-1], m.cell_data["gmsh:physical"][:-1]):
        for tri, tag in zip(block.data, tags):
            fi = face_lookup[tuple(sorted(tri))]
            faces[names[int(tag)]].append(fi)
            boundary_tag[fi] = int(tag)
    np.savez(
        "processed_mesh.npz",
        **{k: mesh_npz[k] for k in mesh_npz.files if k != "boundary_tag"},
        boundary_tag=boundary_tag,
        inlet_faces=np.array(faces["inlet"], dtype=np.int32),
        outlet_faces=np.array(faces["outlet"], dtype=np.int32),
        wall_faces=np.array(faces["walls"], dtype=np.int32),
        object_faces=np.array(faces["object"], dtype=np.int32),
    )
    print({k: len(v) for k, v in faces.items()})
else:
    tag = mesh_npz["boundary_tag"]
    print("boundary tags OK:", {n: int(np.sum(tag == i)) for n, i in
          [("inlet",1),("outlet",2),("walls",3),("object",4)]})

In [ ]:
from les_sphere_flow import SphereLESSolver, SimConfig

# True = faster demo; False = longer wake development
quick = False

if quick:
    cfg = SimConfig(dt=1e-4, t_end=0.05, print_every=50, inlet_ti=0.015)
    max_steps = 200
else:
    cfg = SimConfig(dt=1e-4, t_end=0.2, print_every=100, inlet_ti=0.015)
    max_steps = 800

solver = SphereLESSolver("processed_mesh.npz", cfg)
solver.run(max_steps=max_steps)
outputs = solver.write_all_outputs()
outputs

In [ ]:
from IPython.display import Image, display

for key in ("midplane", "wake", "nut"):
    print(outputs[key])
    display(Image(filename=outputs[key]))